# APY Dataset — Data Preprocessing

## Objective

This notebook prepares the cleaned and quality-flagged APY dataset for exploratory analysis and future machine learning tasks.

The preprocessing workflow will:
- Reshape annual Area, Production, and Yield measurements into long format.
- Standardize the representation of the agricultural year.
- Preserve data-quality flags for traceability.
- Separate aggregate crop totals from individual-crop analysis where appropriate.
- Validate and save the transformed dataset.

The source measurements and their quality flags will be preserved. Missing values will not automatically be replaced with zero, and aggregate categories will not be mixed with individual crops during crop-level comparisons.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path.cwd().parent

input_path = (
    project_root
    / "data"
    / "processed"
    / "historical_food_grains_apy_cleaned_flagged.csv"
)

apy_clean_df = pd.read_csv(input_path)

print("Input file exists:", input_path.exists())
print("Dataset shape:", apy_clean_df.shape)
print("Number of columns:", len(apy_clean_df.columns))
print("\nColumn names:")
print(apy_clean_df.columns.tolist())

Input file exists: True
Dataset shape: (31356, 89)
Number of columns: 89

Column names:
['State', 'District', 'Crop', 'Season', 'Area-2013-14', 'Production-2013-14', 'Yield-2013-14', 'Area-2014-15', 'Production-2014-15', 'Yield-2014-15', 'Area-2015-16', 'Production-2015-16', 'Yield-2015-16', 'Area-2016-17', 'Production-2016-17', 'Yield-2016-17', 'Area-2017-18', 'Production-2017-18', 'Yield-2017-18', 'Area-2018-19', 'Production-2018-19', 'Yield-2018-19', 'Area-2019-20', 'Production-2019-20', 'Yield-2019-20', 'Area-2020-21', 'Production-2020-21', 'Yield-2020-21', 'Area-2021-22', 'Production-2021-22', 'Yield-2021-22', 'Area-2022-23', 'Production-2022-23', 'Yield-2022-23', 'Area-2023-24', 'Production-2023-24', 'Yield-2023-24', 'Area-2024-25', 'Production-2024-25', 'Yield-2024-25', 'Crop_Category', 'QC-2013-14-AreaZero-ProductionPositive', 'QC-2013-14-AreaPositive-ProductionZero', 'QC-2013-14-NegativeMeasurements', 'QC-2013-14-PositiveYieldWithoutPositiveAreaProduction', 'QC-2014-15-AreaZer

In [2]:
# Identify the years represented in the APY dataset.
apy_years = sorted({
    col.split("-")[1] + "-" + col.split("-")[2]
    for col in apy_clean_df.columns
    if col.startswith(("Area-", "Production-", "Yield-"))
    and col.split("-")[1].isdigit()
})

measurement_types = ["Area", "Production", "Yield"]

quality_flag_types = [
    "AreaZero-ProductionPositive",
    "AreaPositive-ProductionZero",
    "NegativeMeasurements",
    "PositiveYieldWithoutPositiveAreaProduction",
]

# Check that every year has all expected measurements and flags.
measurement_check = pd.DataFrame({
    measurement: [
        f"{measurement}-{year}" in apy_clean_df.columns
        for year in apy_years
    ]
    for measurement in measurement_types
}, index=apy_years)

flag_check = pd.DataFrame({
    flag: [
        f"QC-{year}-{flag}" in apy_clean_df.columns
        for year in apy_years
    ]
    for flag in quality_flag_types
}, index=apy_years)

print("Years detected:", apy_years)
print("Number of years:", len(apy_years))

print("\nMeasurement column check:")
display(measurement_check)

print("\nQuality flag column check:")
display(flag_check)

print("\nAll measurement columns present:", measurement_check.all().all())
print("All quality flag columns present:", flag_check.all().all())

Years detected: ['2013-14', '2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']
Number of years: 12

Measurement column check:


,Area,Production,Yield
2013-14,True,True,True
2014-15,True,True,True
2015-16,True,True,True
2016-17,True,True,True
2017-18,True,True,True
2018-19,True,True,True
2019-20,True,True,True
2020-21,True,True,True
2021-22,True,True,True
2022-23,True,True,True



Quality flag column check:


,AreaZero-ProductionPositive,AreaPositive-ProductionZero,NegativeMeasurements,PositiveYieldWithoutPositiveAreaProduction
2013-14,True,True,True,True
2014-15,True,True,True,True
2015-16,True,True,True,True
2016-17,True,True,True,True
2017-18,True,True,True,True
2018-19,True,True,True,True
2019-20,True,True,True,True
2020-21,True,True,True,True
2021-22,True,True,True,True
2022-23,True,True,True,True



All measurement columns present: True
All quality flag columns present: True


In [3]:
# Separate identifier columns from annual measurements and quality flags.
identifier_columns = [
    "State",
    "District",
    "Crop",
    "Season",
    "Crop_Category",
]

# Build one table for each year, then combine them.
apy_long_parts = []

for year in apy_years:
    year_df = apy_clean_df[identifier_columns].copy()
    year_df["Year"] = year

    # Preserve the annual measurements.
    for measurement in ["Area", "Production", "Yield"]:
        year_df[measurement] = apy_clean_df[f"{measurement}-{year}"]

    # Preserve the annual quality flags.
    for flag in quality_flag_types:
        year_df[f"QC_{flag}"] = apy_clean_df[f"QC-{year}-{flag}"]

    apy_long_parts.append(year_df)

apy_long_df = pd.concat(apy_long_parts, ignore_index=True)

print("Original dataset shape:", apy_clean_df.shape)
print("Long-format dataset shape:", apy_long_df.shape)
print("Expected rows:", len(apy_clean_df) * len(apy_years))

display(apy_long_df.head())
display(apy_long_df.dtypes.to_frame("Data Type"))

Original dataset shape: (31356, 89)
Long-format dataset shape: (376272, 13)
Expected rows: 376272


,State,District,Crop,Season,Crop_Category,Year,Area,Production,Yield,QC_AreaZero-ProductionPositive,QC_AreaPositive-ProductionZero,QC_NegativeMeasurements,QC_PositiveYieldWithoutPositiveAreaProduction
0,Andaman And Nicobar Islands,Nicobars,Cereals,Kharif,Grouping — Verify Source Definition,2013-14,0.0,0.0,2377.0,False,False,False,True
1,Andaman And Nicobar Islands,Nicobars,Cereals,Rabi,Grouping — Verify Source Definition,2013-14,0.0,0.0,526.0,False,False,False,True
2,Andaman And Nicobar Islands,Nicobars,Cereals,Total,Grouping — Verify Source Definition,2013-14,0.0,0.0,923.0,False,False,False,True
3,Andaman And Nicobar Islands,Nicobars,Maize,Rabi,Individual Crop Candidate,2013-14,0.0,0.0,526.0,False,False,False,True
4,Andaman And Nicobar Islands,Nicobars,Maize,Total,Individual Crop Candidate,2013-14,0.0,0.0,526.0,False,False,False,True


,Data Type
State,object
District,object
Crop,object
Season,object
Crop_Category,object
Year,object
Area,float64
Production,float64
Yield,float64
QC_AreaZero-ProductionPositive,bool


In [4]:
# Validate the long-format dataset.

expected_rows = len(apy_clean_df) * len(apy_years)

# Each original identifier combination should appear once per year.
long_key = ["State", "District", "Crop", "Season", "Year"]

print("Expected rows:", expected_rows)
print("Actual rows:", len(apy_long_df))
print("Row count correct:", len(apy_long_df) == expected_rows)

print("\nDuplicate long-format keys:",
      apy_long_df.duplicated(subset=long_key).sum())

print("Missing identifier values:",
      apy_long_df[["State", "District", "Crop", "Season", "Year"]]
      .isna().sum().sum())

print("\nMissing measurements:")
display(
    apy_long_df[["Area", "Production", "Yield"]]
    .isna()
    .sum()
    .to_frame("Missing Count")
    .assign(
        Missing_Percent=lambda df:
        (df["Missing Count"] / len(apy_long_df) * 100).round(2)
    )
)

print("\nQuality-flag counts:")
display(
    apy_long_df[
        [col for col in apy_long_df.columns if col.startswith("QC_")]
    ].sum().astype(int).to_frame("Flagged Records")
)

Expected rows: 376272
Actual rows: 376272
Row count correct: True

Duplicate long-format keys: 0
Missing identifier values: 0

Missing measurements:


,Missing Count,Missing_Percent
Area,94183,25.03
Production,95030,25.26
Yield,95033,25.26



Quality-flag counts:


,Flagged Records
QC_AreaZero-ProductionPositive,9358
QC_AreaPositive-ProductionZero,6582
QC_NegativeMeasurements,0
QC_PositiveYieldWithoutPositiveAreaProduction,94235


In [5]:
# Verify that reshaping preserved every annual measurement and quality flag.

measurement_mismatches = {}

for year in apy_years:
    year_rows = apy_long_df.loc[apy_long_df["Year"] == year].reset_index(drop=True)

    for measurement in ["Area", "Production", "Yield"]:
        original_values = apy_clean_df[f"{measurement}-{year}"].reset_index(drop=True)
        reshaped_values = year_rows[measurement]

        if not original_values.equals(reshaped_values):
            measurement_mismatches[f"{measurement}-{year}"] = (
                original_values.compare(reshaped_values).shape[0]
            )

flag_mismatches = {}

for year in apy_years:
    year_rows = apy_long_df.loc[apy_long_df["Year"] == year].reset_index(drop=True)

    for flag in quality_flag_types:
        original_values = apy_clean_df[f"QC-{year}-{flag}"].reset_index(drop=True)
        reshaped_values = year_rows[f"QC_{flag}"]

        if not original_values.equals(reshaped_values):
            flag_mismatches[f"{year}-{flag}"] = (
                original_values.compare(reshaped_values).shape[0]
            )

print("Measurement mismatches:", len(measurement_mismatches))
print("Quality-flag mismatches:", len(flag_mismatches))
print("Measurements preserved:", len(measurement_mismatches) == 0)
print("Quality flags preserved:", len(flag_mismatches) == 0)

Measurement mismatches: 0
Quality-flag mismatches: 0
Measurements preserved: True
Quality flags preserved: True


In [6]:
# Save the validated long-format APY dataset.

output_path = (
    project_root
    / "data"
    / "processed"
    / "historical_food_grains_apy_long_2013-14_to_2024-25.csv"
)

output_path.parent.mkdir(parents=True, exist_ok=True)

apy_long_df.to_csv(output_path, index=False)

# Verify the saved file.
saved_df = pd.read_csv(output_path)

print("Saved dataset:", output_path)
print("File exists:", output_path.exists())
print("Saved shape:", saved_df.shape)
print("Column count:", len(saved_df.columns))

Saved dataset: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\data\processed\historical_food_grains_apy_long_2013-14_to_2024-25.csv
File exists: True
Saved shape: (376272, 13)
Column count: 13


## Preprocessing Summary

### Completed
- Loaded the cleaned, quality-flagged APY dataset.
- Verified all 12 agricultural years and their measurement and quality-flag columns.
- Reshaped the dataset from wide format to long format.
- Preserved Area, Production, Yield, crop classifications, and annual quality flags.
- Validated row counts, identifier completeness, key uniqueness, and measurement preservation.
- Saved the preprocessed dataset separately from the source files.

### Validation Results
- Original records: 31,356
- Agricultural years: 12
- Expected and actual long-format records: 376,272
- Duplicate long-format keys: 0
- Missing identifier values: 0
- Measurement mismatches: 0
- Quality-flag mismatches: 0

### Output
`data/processed/historical_food_grains_apy_long_2013-14_to_2024-25.csv`

### Limitations and Next Steps
Missing measurements and suspicious measurement combinations remain unchanged. Quality flags indicate records that require attention; they do not automatically establish that a record is incorrect. Crop classifications are provisional and should be verified against the source definitions before final modeling decisions.

The next notebook will explore the distribution, trends, relationships, and data-quality patterns in the APY dataset. Aggregate crop categories will be separated from individual-crop comparisons where appropriate.
